# AutoScout24 Audi Q4 e-tron — Italy Primary Scraper

Identical to the Germany scraper except `cy=I` (Italy) is used in the search URL.
Output is written to `outputs/scrape_audi_q4_it_YYYYMMDD.csv`.

Run cells top-to-bottom. The last cell executes the scraper.

In [1]:
import json
import os
import re
from datetime import datetime
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urljoin, urlsplit, urlunsplit
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from playwright.sync_api import TimeoutError as PlaywrightTimeoutError
from playwright.sync_api import sync_playwright


# ---- Italy-specific configuration ------------------------------------------------
SEARCH_URL = (
    "https://www.autoscout24.com/lst/audi/q4-e-tron"
    "?atype=C&cy=I&damaged_listing=exclude&desc=1&ocs_listing=include"
    "&powertype=kw&search_id=wmvs3ql6fl&sort=age&source=homepage_search-mask"
    "&ustate=N%2CU"
)
BASE_URL = "https://www.autoscout24.com"
SCRAPE_DATE = datetime.now().strftime("%Y%m%d")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / f"scrape_audi_q4_it_{SCRAPE_DATE}.csv"

# Set HEADLESS = False temporarily while debugging in a visible browser.
# Optional test mode: export AUTOSCOUT24_MAX_PAGES=1 before running.
HEADLESS = True
PAGE_TIMEOUT_MS = 45_000
POST_COOKIE_WAIT_MS = 1_500
NETWORK_IDLE_TIMEOUT_MS = 1_500
MAX_PAGE_RETRIES = 3
RETRY_WAIT_MS = 2_000
DEBUG_SAMPLE_RECORDS = 2
MAX_PAGES_ENV = os.getenv("AUTOSCOUT24_MAX_PAGES")

USER_AGENT = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

COLUMN_ORDER = [
    "listing_id", "title", "subtitle", "variant", "model_group", "transmission",
    "price", "currency", "price_superscript", "is_conditional_price",
    "mileage", "first_registration", "fuel_or_powertrain", "power_kw", "power_hp",
    "wltp_consumption", "wltp_co2_emissions",
    "seller_name", "seller_type", "dealer_id", "seller_phone",
    "seller_rating_stars", "seller_rating_count",
    "seller_info_url", "seller_imprint_url", "seller_location",
    "country_code", "zip_code", "city", "street",
    "image_count", "primary_image_url", "has_360_image",
    "available_now", "vat_deductible", "delivery_possible",
    "listing_url", "raw_card_text",
]

INTEGER_COLUMNS  = {"image_count", "seller_rating_count"}
FLOAT_COLUMNS    = {"seller_rating_stars"}
BOOLEAN_COLUMNS  = {"has_360_image", "available_now", "vat_deductible",
                    "delivery_possible", "is_conditional_price"}

SELECTORS = {
    "cookie_accept_buttons": [
        "button[data-testid='as24-cmp-accept-all-button']",
        "button[data-testid='as24-cmp-decline-all-button']",
        "button:has-text('Accept All')",
    ],
    "next_data_script": "#__NEXT_DATA__",
    "offer_links": "a[href*='/offers/']",
}

TEXT_MARKERS = {
    "vat_deductible": [
        "VAT deductible", "VAT deduct.", "MwSt. ausweisbar",
        "TVA déductible", "IVA deducibile",
    ],
    "delivery_possible": [
        "Delivery possible", "Nationwide delivery", "Home delivery",
    ],
}

In [2]:
# General helpers
def clean_text(value):
    if value is None:
        return None
    text = re.sub(r"\s+", " ", str(value)).strip()
    return text or None

def first_non_empty(*values):
    for value in values:
        cleaned = clean_text(value)
        if cleaned is not None:
            return cleaned
    return None

def make_absolute_url(url):
    cleaned = clean_text(url)
    if cleaned is None:
        return None
    return urljoin(BASE_URL, cleaned)

def safe_int(value, default=None):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default

def build_results_page_url(base_url, page_number):
    parts = urlsplit(base_url)
    query = dict(parse_qsl(parts.query, keep_blank_values=True))
    if page_number <= 1:
        query.pop("page", None)
    else:
        query["page"] = str(page_number)
    return urlunsplit((parts.scheme, parts.netloc, parts.path,
                       urlencode(query, doseq=True), parts.fragment))

def split_price_and_currency(price_text):
    price_text = clean_text(price_text)
    if price_text is None:
        return None, None
    currency_match = re.match(r"^\D+", price_text)
    currency = clean_text(currency_match.group(0)) if currency_match else None
    numeric_price = clean_text(price_text.replace(currency or "", "", 1))
    return numeric_price, currency

def get_first_phone_number(seller_data):
    for phone in seller_data.get("phones") or []:
        phone_number = clean_text(phone.get("formattedNumber"))
        if phone_number:
            return phone_number
    return None

def parse_wltp_values(listing_data):
    values = listing_data.get("wltpValues") or []
    return (clean_text(values[0]) if len(values) >= 1 else None,
            clean_text(values[1]) if len(values) >= 2 else None)

def parse_power_values(power_text):
    power_text = clean_text(power_text)
    if power_text is None:
        return None, None
    kw_match = re.search(r"([\d.,]+)\s*kW", power_text, flags=re.IGNORECASE)
    hp_match = re.search(r"([\d.,]+)\s*hp", power_text, flags=re.IGNORECASE)
    return (clean_text(kw_match.group(1)) if kw_match else None,
            clean_text(hp_match.group(1)) if hp_match else None)

def build_detail_map(listing_data):
    detail_map = {}
    for item in listing_data.get("vehicleDetails") or []:
        label = clean_text(item.get("ariaLabel"))
        value = clean_text(item.get("data"))
        if label and value:
            detail_map[label] = value
    return detail_map

def build_title(listing_data, dom_title=None):
    dom_title = clean_text(dom_title)
    if dom_title:
        return dom_title
    vehicle = listing_data.get("vehicle") or {}
    parts = [clean_text(vehicle.get(k)) for k in ("make", "model", "modelVersionInput")]
    parts = [p for p in parts if p]
    return clean_text(" ".join(parts)) or clean_text(vehicle.get("variant"))

def build_seller_location(location_data):
    location_data = location_data or {}
    cc = clean_text(location_data.get("countryCode"))
    zp = clean_text(location_data.get("zip"))
    ci = clean_text(location_data.get("city"))
    parts = []
    if cc and zp: parts.append(f"{cc}-{zp}")
    elif cc: parts.append(cc)
    elif zp: parts.append(zp)
    if ci: parts.append(ci)
    return clean_text(" ".join(parts))

def marker_present(raw_text, markers):
    raw_text = clean_text(raw_text)
    if raw_text is None:
        return None
    lowered = raw_text.lower()
    return True if any(m.lower() in lowered for m in markers) else None

def empty_record(listing_url=None, raw_card_text=None):
    record = {column: None for column in COLUMN_ORDER}
    record["listing_url"] = make_absolute_url(listing_url)
    record["raw_card_text"] = clean_text(raw_card_text)
    return record

def print_debug_sample(card_payloads, reason):
    sample = next((item for item in card_payloads.values() if item.get("raw_card_text")), None)
    if sample is None:
        print(f"[debug] {reason}: no sample card text was captured.")
        return
    print(f"[debug] {reason}: sample raw card text")
    print(sample["raw_card_text"][:1_000])

In [3]:
# Browser and session helpers
def launch_browser(playwright):
    browser = playwright.chromium.launch(headless=HEADLESS)
    context = browser.new_context(
        user_agent=USER_AGENT,
        locale="en-GB",
        viewport={"width": 1440, "height": 2200},
    )
    page = context.new_page()
    page.set_default_timeout(PAGE_TIMEOUT_MS)
    return browser, context, page

def handle_cookie_consent(page):
    for selector in SELECTORS["cookie_accept_buttons"]:
        button = page.locator(selector)
        if button.count() == 0:
            continue
        try:
            button.first.click(timeout=3_000)
            print(f"Cookie banner handled with selector: {selector}")
            page.wait_for_timeout(POST_COOKIE_WAIT_MS)
            return True
        except PlaywrightTimeoutError:
            continue
    return False

def wait_for_results_to_load(page):
    page.wait_for_selector(SELECTORS["next_data_script"], state="attached")
    try:
        page.wait_for_selector(SELECTORS["offer_links"], state="attached", timeout=1_000)
    except PlaywrightTimeoutError:
        print("Offer links were not visible yet; continuing with __NEXT_DATA__ only.")
    try:
        page.wait_for_load_state("networkidle", timeout=NETWORK_IDLE_TIMEOUT_MS)
    except PlaywrightTimeoutError:
        pass

def load_next_data(page):
    raw_json = page.locator(SELECTORS["next_data_script"]).inner_text()
    return json.loads(raw_json)

In [4]:
# DOM extraction and card parsing
def extract_listing_card_elements(page, listings):
    listing_paths = [l.get("url") for l in listings if clean_text(l.get("url"))]
    offer_link_count = page.locator(SELECTORS["offer_links"]).count()
    print(f"Found {offer_link_count} DOM offer links on the page.")

    if offer_link_count == 0:
        print("No DOM offer links; using __NEXT_DATA__ only.")
        return {p: {"listing_path": p, "found": False,
                    "dom_title": None, "raw_card_text": None,
                    "raw_card_html": None} for p in listing_paths}

    card_payloads = page.evaluate("""
        (listingPaths) => {
          function findCard(anchor) {
            let current = anchor;
            for (let depth = 0; current && depth < 10; depth += 1, current = current.parentElement) {
              const text = (current.innerText || "").trim();
              if (!text) continue;
              const hasOfferLink = !!current.querySelector('a[href*="/offers/"]');
              const looksLikeCard = /€|\\bkm\\b|\\bkW\\b|\\bhp\\b/i.test(text);
              if (hasOfferLink && looksLikeCard && text.length >= 40) return current;
            }
            return anchor.closest("article, section") || anchor.parentElement || anchor;
          }
          return listingPaths.map((listingPath) => {
            const absoluteUrl = new URL(listingPath, document.baseURI).href;
            const selectors = [
              `a[href="${listingPath}"]`,
              `a[href="${absoluteUrl}"]`,
              `a[href$="${listingPath}"]`,
            ];
            let anchor = null;
            for (const selector of selectors) {
              anchor = document.querySelector(selector);
              if (anchor) break;
            }
            if (!anchor) return {listing_path: listingPath, found: false,
                                  dom_title: null, raw_card_text: null, raw_card_html: null};
            const card = findCard(anchor);
            const heading = card.querySelector("h1, h2, h3");
            return {
              listing_path: listingPath, found: true,
              dom_title: (heading?.textContent || anchor.textContent || "").trim() || null,
              raw_card_text: (card.innerText || "").trim() || null,
              raw_card_html: (card.outerHTML || "").trim().slice(0, 8000) || null,
            };
          });
        }
    """, listing_paths)

    payload_by_path = {item["listing_path"]: item for item in card_payloads}
    matched = sum(1 for item in card_payloads if item.get("found"))
    print(f"Matched {matched}/{len(listing_paths)} DOM cards to listings.")
    return payload_by_path


def parse_one_listing_card(listing_data, card_payload):
    vehicle = listing_data.get("vehicle") or {}
    seller = listing_data.get("seller") or {}
    price_data = listing_data.get("price") or {}
    ratings = listing_data.get("ratings") or {}
    location = listing_data.get("location") or {}
    detail_map = build_detail_map(listing_data)
    raw_card_text = clean_text((card_payload or {}).get("raw_card_text"))
    wltp_consumption, wltp_co2_emissions = parse_wltp_values(listing_data)

    title = build_title(listing_data, dom_title=(card_payload or {}).get("dom_title"))
    subtitle = first_non_empty(vehicle.get("subtitle"),
                               vehicle.get("modelVersionInput"),
                               vehicle.get("variant"))
    price_text = first_non_empty(price_data.get("priceFormatted"))
    price, currency = split_price_and_currency(price_text)
    power_text = first_non_empty(detail_map.get("Power"), raw_card_text)
    power_kw, power_hp = parse_power_values(power_text)

    delivery_hint = clean_text(
        (((seller.get("dealer") or {}).get("nationwideListingsData") or {}).get("consumerHint"))
    )
    delivery_possible = True if delivery_hint else marker_present(raw_card_text, TEXT_MARKERS["delivery_possible"])
    vat_deductible = marker_present(raw_card_text, TEXT_MARKERS["vat_deductible"])

    return {
        "listing_id": clean_text(listing_data.get("id")),
        "title": title, "subtitle": subtitle,
        "variant": clean_text(vehicle.get("variant")),
        "model_group": clean_text(vehicle.get("modelGroup")),
        "transmission": first_non_empty(detail_map.get("Gear"), vehicle.get("transmission")),
        "price": price, "currency": currency,
        "price_superscript": clean_text(price_data.get("priceSuperscriptString")),
        "is_conditional_price": price_data.get("isConditionalPrice"),
        "mileage": first_non_empty(detail_map.get("Mileage"), vehicle.get("mileageInKm")),
        "first_registration": first_non_empty(detail_map.get("First registration"),
                                              detail_map.get("Registration date")),
        "fuel_or_powertrain": first_non_empty(detail_map.get("Fuel type"), vehicle.get("fuel")),
        "power_kw": power_kw, "power_hp": power_hp,
        "wltp_consumption": wltp_consumption, "wltp_co2_emissions": wltp_co2_emissions,
        "seller_name": first_non_empty(seller.get("companyName"), seller.get("contactName")),
        "seller_type": clean_text(seller.get("type")),
        "dealer_id": clean_text(seller.get("id")),
        "seller_phone": get_first_phone_number(seller),
        "seller_rating_stars": ratings.get("ratingsStars"),
        "seller_rating_count": ratings.get("ratingsCount"),
        "seller_info_url": make_absolute_url(((seller.get("links") or {}).get("infoPage"))),
        "seller_imprint_url": make_absolute_url(((seller.get("links") or {}).get("imprint"))),
        "seller_location": build_seller_location(location),
        "country_code": clean_text(location.get("countryCode")),
        "zip_code": clean_text(location.get("zip")),
        "city": clean_text(location.get("city")),
        "street": clean_text(location.get("street")),
        "image_count": len(listing_data.get("images") or []),
        "primary_image_url": make_absolute_url((listing_data.get("images") or [None])[0]),
        "has_360_image": listing_data.get("has360Image"),
        "available_now": listing_data.get("availableNow"),
        "vat_deductible": vat_deductible,
        "delivery_possible": delivery_possible,
        "listing_url": make_absolute_url(listing_data.get("url")),
        "raw_card_text": raw_card_text,
    }

In [5]:
# Page loading, pagination, and deduplication
def parse_listings_into_records(listings, card_payload_by_path, page_number):
    records = []
    for listing in listings:
        listing_path = listing.get("url")
        card_payload = card_payload_by_path.get(listing_path, {})
        try:
            record = parse_one_listing_card(listing, card_payload)
        except Exception as exc:
            print(f"[page {page_number}] Failed to parse listing {listing_path}: {exc}")
            record = empty_record(listing_url=listing_path,
                                  raw_card_text=card_payload.get("raw_card_text"))
        records.append(record)
    return records


def load_results_page(page, page_number):
    target_url = build_results_page_url(SEARCH_URL, page_number)
    last_exception = None

    for attempt in range(1, MAX_PAGE_RETRIES + 1):
        try:
            print(f"[page {page_number}] Loading {target_url} (attempt {attempt}/{MAX_PAGE_RETRIES})")
            response = page.goto(target_url, wait_until="domcontentloaded")
            if response is not None and response.status >= 400:
                raise RuntimeError(f"HTTP {response.status} on {target_url}")
            handle_cookie_consent(page)
            wait_for_results_to_load(page)
            next_data = load_next_data(page)
            page_props = next_data["props"]["pageProps"]
            listings = page_props.get("listings") or []
            if not listings:
                raise RuntimeError("No listings found in __NEXT_DATA__.")
            card_payload_by_path = extract_listing_card_elements(page, listings)
            records = parse_listings_into_records(listings, card_payload_by_path, page_number)
            print(f"[page {page_number}] Parsed {len(records)} records.")
            return {
                "page_number": page_number, "page_url": target_url,
                "total_pages": safe_int(page_props.get("numberOfPages"), default=1),
                "total_results": safe_int(page_props.get("numberOfResults")),
                "records": records,
            }
        except Exception as exc:
            last_exception = exc
            print(f"[page {page_number}] Attempt {attempt}/{MAX_PAGE_RETRIES} failed: {exc}")
            if attempt < MAX_PAGE_RETRIES:
                page.wait_for_timeout(RETRY_WAIT_MS)

    raise RuntimeError(f"Page {page_number} failed after {MAX_PAGE_RETRIES} attempts.") from last_exception


def deduplicate_records(records):
    unique_records, seen_urls, dup_count = [], set(), 0
    for record in records:
        url = record.get("listing_url")
        if url and url in seen_urls:
            dup_count += 1
            continue
        if url:
            seen_urls.add(url)
        unique_records.append(record)
    print(f"Removed {dup_count} duplicate records.")
    return unique_records


def _scrape_search_results_sync():
    all_records, failed_pages = [], []
    page_limit = safe_int(MAX_PAGES_ENV)

    with sync_playwright() as playwright:
        browser, context, page = launch_browser(playwright)
        try:
            first_page_data = load_results_page(page, page_number=1)
            total_pages = first_page_data["total_pages"]
            total_results = first_page_data["total_results"]
            if page_limit is not None and page_limit > 0:
                total_pages = min(total_pages, page_limit)
                print(f"Page limit override: scraping {total_pages} page(s).")
            print(f"Italy: {total_results} results across {total_pages} pages.")
            all_records.extend(first_page_data["records"])
            for page_number in range(2, total_pages + 1):
                try:
                    page_data = load_results_page(page, page_number=page_number)
                except Exception as exc:
                    print(f"[page {page_number}] Skipping: {exc}")
                    failed_pages.append(page_number)
                    continue
                all_records.extend(page_data["records"])
        finally:
            context.close()
            browser.close()

    print(f"Collected {len(all_records)} raw records before de-duplication.")
    all_records = deduplicate_records(all_records)
    if all_records:
        print("Sample parsed records:")
        print(json.dumps(all_records[:DEBUG_SAMPLE_RECORDS], indent=2, ensure_ascii=False))
    return all_records, failed_pages


def scrape_search_results():
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(_scrape_search_results_sync).result()


def create_dataframe(records):
    return pd.DataFrame(records).reindex(columns=COLUMN_ORDER)


def clean_and_normalize_dataframe(dataframe):
    dataframe = dataframe.copy()
    for column in dataframe.columns:
        if column in INTEGER_COLUMNS or column in FLOAT_COLUMNS or column in BOOLEAN_COLUMNS:
            continue
        dataframe[column] = dataframe[column].apply(clean_text)
    for column in INTEGER_COLUMNS:
        if column in dataframe.columns:
            dataframe[column] = pd.to_numeric(dataframe[column], errors="coerce").astype("Int64")
    for column in FLOAT_COLUMNS:
        if column in dataframe.columns:
            dataframe[column] = pd.to_numeric(dataframe[column], errors="coerce")
    for column in BOOLEAN_COLUMNS:
        if column in dataframe.columns:
            dataframe[column] = dataframe[column].astype("boolean")
    return dataframe.where(pd.notna(dataframe), None)


def save_to_csv(dataframe, output_path):
    dataframe.to_csv(output_path, index=False, encoding="utf-8")
    print(f"Saved {len(dataframe)} rows to {output_path}")

In [6]:
# Run the scraper
scraped_records, failed_pages = scrape_search_results()
scraped_dataframe = create_dataframe(scraped_records)
scraped_dataframe = clean_and_normalize_dataframe(scraped_dataframe)
save_to_csv(scraped_dataframe, OUTPUT_CSV)

if failed_pages:
    print(f"Pages skipped after retries: {failed_pages}")
else:
    print("All result pages scraped successfully.")

print(f"\nTotal Italy listings scraped: {len(scraped_dataframe)}")
scraped_dataframe.head()

[page 1] Loading https://www.autoscout24.com/lst/audi/q4-e-tron?atype=C&cy=I&damaged_listing=exclude&desc=1&ocs_listing=include&powertype=kw&search_id=wmvs3ql6fl&sort=age&source=homepage_search-mask&ustate=N%2CU (attempt 1/3)


Cookie banner handled with selector: button[data-testid='as24-cmp-accept-all-button']


Found 40 DOM offer links on the page.
Matched 20/20 DOM cards to listings.
[page 1] Parsed 20 records.
Italy: 227 results across 12 pages.
[page 2] Loading https://www.autoscout24.com/lst/audi/q4-e-tron?atype=C&cy=I&damaged_listing=exclude&desc=1&ocs_listing=include&powertype=kw&search_id=wmvs3ql6fl&sort=age&source=homepage_search-mask&ustate=N%2CU&page=2 (attempt 1/3)


Offer links were not visible yet; continuing with __NEXT_DATA__ only.


Found 0 DOM offer links on the page.
No DOM offer links; using __NEXT_DATA__ only.
[page 2] Parsed 20 records.
[page 3] Loading https://www.autoscout24.com/lst/audi/q4-e-tron?atype=C&cy=I&damaged_listing=exclude&desc=1&ocs_listing=include&powertype=kw&search_id=wmvs3ql6fl&sort=age&source=homepage_search-mask&ustate=N%2CU&page=3 (attempt 1/3)


...

Offer links were not visible yet; continuing with __NEXT_DATA__ only.


Found 0 DOM offer links on the page.
No DOM offer links; using __NEXT_DATA__ only.
[page 11] Parsed 20 records.
[page 12] Loading https://www.autoscout24.com/lst/audi/q4-e-tron?atype=C&cy=I&damaged_listing=exclude&desc=1&ocs_listing=include&powertype=kw&search_id=wmvs3ql6fl&sort=age&source=homepage_search-mask&ustate=N%2CU&page=12 (attempt 1/3)


Offer links were not visible yet; continuing with __NEXT_DATA__ only.


Found 0 DOM offer links on the page.
No DOM offer links; using __NEXT_DATA__ only.
[page 12] Parsed 12 records.
Collected 232 raw records before de-duplication.
Removed 5 duplicate records.
Sample parsed records:
[
  {
    "listing_id": "8d7209f6-394c-454f-a474-b673e6502204",
    "title": "Audi Q4 e-tron 45 s line edition quattro 265cv",
    "subtitle": "45 s line edition quattro 265cv",
    "variant": "Q4 e-tron 45",
    "model_group": "Q4 e-tron",
    "transmission": "Automatic",
    "price": "36,900",
    "currency": "€",
    "price_superscript": null,
    "is_conditional_price": false,
    "mileage": "48,410 km",
    "first_registration": "10/2023",
    "fuel_or_powertrain": "Electric",
    "power_kw": "195",
    "power_hp": "265",
    "wltp_consumption": null,
    "wltp_co2_emissions": null,
    "seller_name": "Autocentri Balduina Srl",
    "seller_type": "Dealer",
    "dealer_id": "4198",
    "seller_phone": "+39 06 - 78461411",
    "seller_rating_stars": 5,
    "seller_rating_co

,listing_id,title,subtitle,variant,model_group,transmission,price,currency,price_superscript,is_conditional_price,...,city,street,image_count,primary_image_url,has_360_image,available_now,vat_deductible,delivery_possible,listing_url,raw_card_text
0,8d7209f6-394c-454f-a474-b673e6502204,Audi Q4 e-tron 45 s line edition quattro 265cv,45 s line edition quattro 265cv,Q4 e-tron 45,Q4 e-tron,Automatic,"36,900",€,None,False,...,Roma - Rm,"Via Appia Nuova, 803",15,https://prod.pictures.autoscout24.net/listing-...,<NA>,True,<NA>,<NA>,https://www.autoscout24.com/offers/audi-q4-e-t...,Audi Q4 e-tron 45 s line edition quattro 265cv...
1,384baec8-934a-47d7-ae1c-656829b6e5d5,Audi Q4 e-tron Q4 sportback e-tron 40 s line e...,Q4 sportback e-tron 40 s line edition,Q4 e-tron Sportback,Q4 e-tron,Automatic,"35,900",€,1,False,...,Roma - Rm,"Via Appia Nuova, 803",13,https://prod.pictures.autoscout24.net/listing-...,<NA>,True,<NA>,<NA>,https://www.autoscout24.com/offers/audi-q4-e-t...,Audi Q4 e-tron Q4 sportback e-tron 40 s line e...
2,48e267d1-5179-43cb-8876-0b527f813493,Audi Q4 e-tron 35 business advanced,35 business advanced,Q4 e-tron 35,Q4 e-tron,Automatic,"39,900",€,1,False,...,Roma - Rm,"Via Appia Nuova, 803",16,https://prod.pictures.autoscout24.net/listing-...,<NA>,True,<NA>,<NA>,https://www.autoscout24.com/offers/audi-q4-e-t...,Audi Q4 e-tron 35 business advanced Add to lis...
3,45ae23cf-7af8-43d1-94bc-3ea564b09546,Audi Q4 e-tron Q4 Sportback e-tron 35,Q4 Sportback e-tron 35,Q4 e-tron Sportback,Q4 e-tron,Automatic,"27,900",€,1,True,...,Ponte S. Giovanni - Perugia - Pg,Via della Valtiera 7,15,https://prod.pictures.autoscout24.net/listing-...,<NA>,True,<NA>,<NA>,https://www.autoscout24.com/offers/audi-q4-e-t...,Audi Q4 e-tron Q4 Sportback e-tron 35 Add to l...
4,eccb50be-99d8-4dd3-b390-f68b83f0b638,Audi Q4 e-tron sportback e-tron 45 business,sportback e-tron 45 business,Q4 e-tron Sportback,Q4 e-tron,Automatic,"43,900",€,1,False,...,Padova - Pd,"Viale Della Navigazione Interna, 60",15,https://prod.pictures.autoscout24.net/listing-...,<NA>,True,<NA>,<NA>,https://www.autoscout24.com/offers/audi-q4-e-t...,Audi Q4 e-tron sportback e-tron 45 business Ad...
